# Langgraph - Hello World

Langgraph is an open-source AI agent framework, built by **LangChain**, designed for creating, deploying, and managing complex generative AI agent workflows. It stands out by using a graph-based architecthre to model the intricate relationships and flow of information within an AI system.

Langgraph builds AI workflows as a graph of steps (nodes), where the flow between steps (edges) can branch, loop, or end based on dynamic decisions. Each step can access and update a shared state, allowing for complex, stateful, and iterative agent workflows.

![[source](https://www.ionio.ai/blog/a-comprehensive-guide-about-langgraph-code-included)](assets/img/01-lggrph.webp)

For example:

* Nodes (circles) are the individual steps or agents (e.g., LLM calls, tool invocations).
* Conditional edges (orange boxes) decide which path to follow vased on logic or the current state.
* Arrows show the flow of information and control.
* Diamonds labeled END are exit points — where workflow finishes.

We create workflows using these elements.

> Workflows are systems where LLMs and tools are orchestrated through predefined code paths. Agents, on the other hand, are systems where LLMs dynamically direct their own processes and tool usage, maintaining control over how they accomplish tasks.

![[source](https://langchain-ai.github.io/langgraph/tutorials/workflows/)](assets/img/01-workflow-agents.webp)

In classic workflows (left), the developer fully defines the sequence or branching. The LLM just fills in outputs; it **does note** control routing or structure.

* *Examples*: Standard pipelines or parallel tasks with static flows.

On the other hand, dynamic workflows (center) are built so that the **LLM itself makes decisions about control flow**.

At certain points (like the "Router" or "Orchestrator"), the LLM gets to choose which path to take or which components to activate.  The structure is flexible, and the path taken may change at runtime, **not strictly predefined** by the developer.

* *Examples*: An LLM acting as a router decides shich specialist model handles an input. Or, an orchestrator LLM assigns tasks dinamically.

An agent (right) makdes decisions and acts, often with tools and in a loop. It observes the environment, gets feedback, and then decides the next action — iteratively and autonomously.

Now, let's intall the `langgraph` library and create a "Hello World" example.

```shell
pip install langgraph
```

Also, you may need to install langchain as well.

```shell
pip install langchain
```

I will use OpenAI models, so I must also install the `langchain-openai` package.

```shell
pip install langchain-openai
```

We should set our OpenAI key.

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()

True

Supose we want our agent to answer weather-related questions with a custom function.

In [18]:
from langgraph.prebuilt import create_react_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    if city.lower() == "fortaleza":
        return "Fortaleza is always warm!"
    return f"The current weather in {city} is sunny with a temperature of 25°C."

agent = create_react_agent(
    model="gpt-3.5-turbo",
    tools=[get_weather],
    prompt="You are a helpful assistant"
)

agent.invoke({"messages": [{"role": "user", "content": "What's the weather like in Fortaleza?"}]})

/tmp/ipykernel_47392/1604561672.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


{'messages': [HumanMessage(content="What's the weather like in Fortaleza?", additional_kwargs={}, response_metadata={}, id='636ae75a-e77f-4864-ba3e-b23ccf68e2a3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 60, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DL22jHAwHyL9qQZMhBMN3aI6e6llD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d04fa-60b4-7ba3-9bce-cf1f4e1de783-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Fortaleza'}, 'id': 'call_7i02zzXs1mquBEwvG643UNuu', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_to

In this setup, the `get_weather` function serves as simple tool that the agent can call when a user asks about the weather in a city.

The `create_react_agent` function from LangGraph registers this tool and provides an initial prompt to shape the assistant's behavior.

When we invoke the agent with a message like "what is the weather in sf", several steps happen behind the scenes:

1. **Message Parsing**: The agent parses the user's request and determines that a tool call is needed.

1. **Function Calling**: It extracts the city ("San Francisco"), calls `get_weather("San Francisco")`, and gets the response.

1. **Response Generation**: The agent then incorporates the tool's output into its final reply:  
*"The weather in Fortaleza is always warm!"*

To design an AI agent that can reason about when to call custom tools — like a weather API — **LangGraph** allows us to construct modular, state-driven workflows.

By defining nodes for language generation and tool invocation, and then linking them with conditional logic, we can create an agent that flexibly moves between "thinking" and "acting".

In [19]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [20]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

graph = StateGraph(State)

In [21]:
@tool
def get_weather(location: str) -> str:
    """Call to get the current weather"""
    if location.lower() in ["yorkshire"]:
        return "It's cold and wet."
    else:
        return "It's warm and sunny."
    
llm = ChatOpenAI(model="gpt-4o-mini")

tools = [get_weather]

llm_with_tools = llm.bind_tools(tools)

tool_node = ToolNode(tools)
graph.add_node("tool_node", tool_node)

In [22]:
def prompt_node(state: State) -> State:
    new_message = llm_with_tools.invoke(state["messages"])
    return {"messages": [new_message]}

graph.add_node("prompt_node", prompt_node)

In [23]:
def conditional_edge(state: State) -> Literal['tool_node', '__end__']:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tool_node"
    else:
        return "__end__"
    
graph.add_conditional_edges(
    "prompt_node",
    conditional_edge,
)

graph.add_edge("tool_node", "prompt_node")
graph.set_entry_point("prompt_node")

In [24]:
APP = graph.compile()

new_state = APP.invoke({"messages": ["What's the weather in Yorkshire?"]})

print(new_state["messages"][-1].content)

The weather in Yorkshire is cold and wet.


With this setup, the workflow follows these steps:

1. **Initialization**: The user's question is entered into the state as a message.

1. **Prompt Node:** The agent, enhanced with tool awareness, considers if it needs external help (by calling a tool) or can respond directly.
1. **Conditional Edge**: If the LLM decides to call a tool (detected via `tool_calls`), the workflow routes to the tool node; otherwise, it ends.
1. **Tool Node**: The relevant tool — like `get_weather` — is invoked with extracted parameters.
1. **Loop/Finish**: The result is passed back to the prompt node for further reasoning or, if no more tool calls are required, the workflow ends.


If you’re having trouble following what’s happening here, don’t worry — this is just some code to get things running. As you continue through this series, everything will become clear.